# Airfoil Spatiotemporal Forecasting (Kaggle, 2×T4)
Production-grade notebook for large-scale CFD forecasting with chunked CSV loading, memmaps, shared spatial stage, and temporal head comparison.


## Section A — Config & profiles


In [ ]:
from __future__ import annotations
import os, gc, math, time, json, copy, random, hashlib, warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.decomposition import IncrementalPCA, PCA
from scipy.spatial import cKDTree

@dataclass
class Config:
    "Dataclass config for data, profile, training, memory, and outputs."
    pressure_csv: str = '/kaggle/input/airfoil-cfd/pressure.csv'
    u_csv: str = '/kaggle/input/airfoil-cfd/u_velocity.csv'
    v_csv: str = '/kaggle/input/airfoil-cfd/v_velocity.csv'
    profile: str = 'balanced'
    train_seconds_list: List[float] = field(default_factory=lambda: [1.0, 3.0, 5.0])
    total_duration_seconds: float = 10.0
    dt_override: Optional[float] = None
    predict_rest: bool = True
    model_list: List[str] = field(default_factory=lambda: ['lstm', 'transformer', 'bi_st_mamba'])
    training_mode: str = 'joint'  # joint | per_variable
    forecast_mode: str = 'direct'  # direct | autoregressive
    tin: int = 32
    tout: int = 16
    stride: int = 1
    pca_mode: str = 'incremental'  # incremental | randomized
    n_modes: int = 64
    explained_variance_threshold: Optional[float] = None
    max_modes_cap: int = 256
    pca_batch_t: int = 16
    recon_error_threshold: float = 0.2
    graph_mode: str = 'knn'
    knn_k: int = 12
    radius: float = 0.01
    mutual_knn: bool = True
    direct_pod_input: bool = True
    spatial_projected_input: bool = False
    include_current_fields_in_nodes: bool = False
    include_static_geometry_placeholder: bool = True
    spatial_hidden: int = 128
    spatial_latent_dim: int = 128
    spatial_layers: int = 2
    batch_size_per_gpu: int = 16
    epochs: int = 30
    patience: int = 6
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_accum: int = 2
    grad_clip: float = 1.0
    scheduler: str = 'cosine'
    num_workers: int = 2
    persistent_workers: bool = True
    pin_memory: bool = True
    amp: bool = True
    fp_dtype: str = 'float32'
    chunk_rows: int = 50_000
    debug_downsample_cells: Optional[int] = None
    val_fraction_of_train: float = 0.2
    use_robust_scaling: bool = False
    teacher_forcing: float = 1.0
    scheduled_sampling_decay: float = 0.95
    output_dir: str = './outputs/airfoil_st'
    checkpoint_dir: str = './checkpoints/airfoil_st'
    cache_dir: str = './cache/airfoil_st'
    seed: int = 42
    deterministic: bool = True

def apply_profile(cfg: Config) -> Config:
    "Apply debug/balanced/full profile defaults."
    cfg = copy.deepcopy(cfg)
    if cfg.profile == 'debug':
        cfg.epochs = 3; cfg.patience = 2; cfg.batch_size_per_gpu = 4; cfg.chunk_rows = 20_000
        cfg.n_modes = 16; cfg.debug_downsample_cells = 50_000; cfg.train_seconds_list = [1.0]
        cfg.num_workers = 0; cfg.persistent_workers = False
    elif cfg.profile == 'balanced':
        cfg.batch_size_per_gpu = 16; cfg.chunk_rows = 50_000; cfg.n_modes = 64
    elif cfg.profile == 'full':
        cfg.epochs = 50; cfg.patience = 10; cfg.batch_size_per_gpu = 16; cfg.chunk_rows = 100_000; cfg.n_modes = 96; cfg.num_workers = 4
    else:
        raise ValueError(f'Unknown profile {cfg.profile}')
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    assert cfg.direct_pod_input != cfg.spatial_projected_input, "Config error: exactly one of 'direct_pod_input' or 'spatial_projected_input' must be True."
    return cfg

def config_hash(cfg: Config) -> str:
    "Hash config for stale-cache-safe keys."
    return hashlib.sha256(json.dumps(asdict(cfg), sort_keys=True, default=str).encode()).hexdigest()[:16]

cfg = apply_profile(Config())

def validate_input_paths(cfg: Config) -> None:
    "Warn clearly when expected input CSV paths are missing."
    missing = [p for p in [cfg.pressure_csv, cfg.u_csv, cfg.v_csv] if not Path(p).exists()]
    if missing:
        warnings.warn('Missing input CSV files: ' + ', '.join(missing) + '. Update Config paths for your environment.')

validate_input_paths(cfg)
print('Profile:', cfg.profile)
print('Config hash:', config_hash(cfg))


## Section B — Environment & GPU setup


In [ ]:
def set_global_seed(seed: int, deterministic: bool = True) -> None:
    "Set reproducible seeds for python/numpy/torch/cuda."
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try: torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception: pass

def print_env() -> None:
    "Print torch/cuda runtime and visible GPUs."
    print('torch:', torch.__version__)
    print('cuda:', torch.cuda.is_available(), 'version:', torch.version.cuda)
    print('visible_gpus:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  gpu{i}: {p.name}, {p.total_memory/1024**3:.1f} GB')

def maybe_dp(model: nn.Module) -> nn.Module:
    "Wrap with DataParallel for 2xT4 runtime."
    if torch.cuda.is_available() and torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_global_seed(cfg.seed, cfg.deterministic)
print_env()
scaler = GradScaler(enabled=(cfg.amp and device.type == 'cuda'))
print('AMP enabled:', scaler.is_enabled())


## Section C — Data I/O pipeline (large-scale)


In [ ]:
def count_rows(path: str) -> int:
    "Count CSV rows excluding header."
    with open(path, 'r', newline='') as f: return sum(1 for _ in f) - 1

def infer_timesteps(path: str) -> int:
    "Infer number of timestep columns from CSV header."
    h = pd.read_csv(path, nrows=0); assert len(h.columns) >= 3
    return len(h.columns) - 2

def mm_dtype(cfg: Config):
    "Resolve dtype used for memmaps."
    return np.float32 if cfg.fp_dtype == 'float32' else np.float64

def load_csv_to_memmap(path: str, out_values: str, out_coords: Optional[str]=None, chunk_rows: int=50_000, dtype: Any=np.float32, max_rows: Optional[int]=None) -> Tuple[Tuple[int,int], Optional[Tuple[int,int]]]:
    "Stream CSV into memmaps: values (N,T), optional coords (N,2)."
    n_rows = count_rows(path); n_t = infer_timesteps(path)
    if max_rows is not None: n_rows = min(n_rows, max_rows)
    v_mm = np.memmap(out_values, mode='w+', dtype=dtype, shape=(n_rows, n_t))
    c_mm = np.memmap(out_coords, mode='w+', dtype=dtype, shape=(n_rows,2)) if out_coords else None
    ptr = 0
    for chunk in tqdm(pd.read_csv(path, chunksize=chunk_rows), desc=f'load {Path(path).name}'):
        arr = chunk.to_numpy(dtype=dtype, copy=False)
        if max_rows is not None and ptr >= max_rows: break
        if max_rows is not None: arr = arr[:max_rows-ptr]
        n = len(arr)
        if n == 0: break
        assert arr.shape[1] == n_t + 2, f"CSV format mismatch in {path}: expected {n_t+2} columns ([x,y]+timesteps), got {arr.shape[1]}"
        if c_mm is not None: c_mm[ptr:ptr+n] = arr[:, :2]
        v_mm[ptr:ptr+n] = arr[:, 2:]
        ptr += n
    assert ptr == n_rows, f'Data loading mismatch: wrote {ptr} rows but expected {n_rows}'
    v_mm.flush();
    if c_mm is not None: c_mm.flush()
    del v_mm, c_mm
    return (n_rows, n_t), ((n_rows,2) if out_coords else None)

def sampled_coord_check(a: np.ndarray, b: np.ndarray, n_samples: int=10_000, atol: float=1e-6) -> None:
    "Sampled coordinate consistency check."
    n = a.shape[0]; idx = np.linspace(0, n-1, min(n_samples, n), dtype=np.int64)
    assert np.allclose(a[idx], b[idx], atol=atol), 'coordinate mismatch sampled'

def full_coord_check_from_csv(path_ref: str, path_other: str, chunk_rows: int=200_000, atol: float=1e-6, max_rows: Optional[int]=None) -> None:
    "Optional full coordinate consistency check across CSVs in chunks."
    r_iter = pd.read_csv(path_ref, usecols=[0,1], chunksize=chunk_rows)
    o_iter = pd.read_csv(path_other, usecols=[0,1], chunksize=chunk_rows)
    seen = 0
    for r, o in tqdm(zip(r_iter, o_iter), desc=f'coord_full_{Path(path_other).name}'):
        ra = r.to_numpy(dtype=np.float64, copy=False)
        oa = o.to_numpy(dtype=np.float64, copy=False)
        if max_rows is not None and seen >= max_rows: break
        if max_rows is not None:
            keep = max_rows - seen
            ra = ra[:keep]; oa = oa[:keep]
        assert ra.shape == oa.shape, f'Coordinate chunk shape mismatch: {ra.shape} vs {oa.shape}'
        if not np.allclose(ra, oa, atol=atol): raise AssertionError('full coordinate mismatch')
        seen += len(ra)

def sampled_coord_check_from_csv(path_ref: str, path_other: str, n_samples: int=10_000, max_rows: Optional[int]=None, atol: float=1e-6) -> None:
    "Sample coordinate rows from two CSVs and compare x/y."
    n = count_rows(path_ref)
    if max_rows is not None: n = min(n, max_rows)
    idx = set(np.linspace(0, n-1, min(n_samples, n), dtype=np.int64).tolist())
    ref_xy = []
    oth_xy = []
    for i, (r,o) in enumerate(zip(pd.read_csv(path_ref, usecols=[0,1], chunksize=1), pd.read_csv(path_other, usecols=[0,1], chunksize=1))):
        if i in idx:
            ref_xy.append(r.to_numpy(dtype=np.float64, copy=False)[0])
            oth_xy.append(o.to_numpy(dtype=np.float64, copy=False)[0])
        if i >= n-1: break
    ref_xy = np.asarray(ref_xy); oth_xy = np.asarray(oth_xy)
    assert ref_xy.shape == oth_xy.shape
    assert np.allclose(ref_xy, oth_xy, atol=atol), f'coordinate mismatch sampled: {Path(path_other).name}'

def cache_key(*parts: Any) -> str:
    "Build deterministic cache key from config/split/model mode."
    return hashlib.sha256('::'.join(map(str, parts)).encode()).hexdigest()[:16]

io_key = cache_key('io', config_hash(cfg), cfg.pressure_csv, cfg.u_csv, cfg.v_csv, cfg.profile)
io_dir = Path(cfg.cache_dir) / f'io_{io_key}'; io_dir.mkdir(parents=True, exist_ok=True)
coords_path = str(io_dir/'coords.mm'); p_path = str(io_dir/'pressure.mm'); u_path = str(io_dir/'u.mm'); v_path = str(io_dir/'v.mm')
meta_path = io_dir/'meta.json'

if not meta_path.exists():
    dt = mm_dtype(cfg)
    p_shape, c_shape = load_csv_to_memmap(cfg.pressure_csv, p_path, coords_path, cfg.chunk_rows, dt, cfg.debug_downsample_cells)
    u_shape, _ = load_csv_to_memmap(cfg.u_csv, u_path, None, cfg.chunk_rows, dt, cfg.debug_downsample_cells)
    v_shape, _ = load_csv_to_memmap(cfg.v_csv, v_path, None, cfg.chunk_rows, dt, cfg.debug_downsample_cells)
    assert p_shape == u_shape == v_shape, 'shape mismatch p/u/v'

    # Required coordinate consistency across all three files
    sampled_coord_check_from_csv(cfg.pressure_csv, cfg.u_csv, n_samples=5000, max_rows=cfg.debug_downsample_cells)
    sampled_coord_check_from_csv(cfg.pressure_csv, cfg.v_csv, n_samples=5000, max_rows=cfg.debug_downsample_cells)
    if cfg.profile == 'debug':
        full_coord_check_from_csv(cfg.pressure_csv, cfg.u_csv, chunk_rows=cfg.chunk_rows, max_rows=cfg.debug_downsample_cells)
        full_coord_check_from_csv(cfg.pressure_csv, cfg.v_csv, chunk_rows=cfg.chunk_rows, max_rows=cfg.debug_downsample_cells)

    meta_path.write_text(json.dumps({'dtype': str(np.dtype(dt)), 'n_cells': p_shape[0], 'n_timesteps': p_shape[1], 'coords_shape': c_shape, 'coords_path': coords_path, 'pressure_path': p_path, 'u_path': u_path, 'v_path': v_path, 'io_key': io_key}, indent=2))

meta = json.loads(meta_path.read_text())
N_CELLS = int(meta['n_cells']); N_TIMESTEPS = int(meta['n_timesteps']); DTYPE = np.dtype(meta['dtype'])
coords_mm = np.memmap(meta['coords_path'], mode='r', dtype=DTYPE, shape=(N_CELLS,2))
p_mm = np.memmap(meta['pressure_path'], mode='r', dtype=DTYPE, shape=(N_CELLS,N_TIMESTEPS))
u_mm = np.memmap(meta['u_path'], mode='r', dtype=DTYPE, shape=(N_CELLS,N_TIMESTEPS))
v_mm = np.memmap(meta['v_path'], mode='r', dtype=DTYPE, shape=(N_CELLS,N_TIMESTEPS))
print('memmaps:', (N_CELLS, N_TIMESTEPS), DTYPE)


## Section D — Time metadata


In [ ]:
@dataclass
class SplitDef:
    "Split boundaries for one experiment run."
    train_end: int
    val_end: int
    test_end: int
    train_seconds: float

def infer_dt(cfg: Config, n_t: int) -> float:
    "Infer dt from total duration and timesteps with optional override."
    if cfg.dt_override is not None: return float(cfg.dt_override)
    if n_t <= 1: raise ValueError(f'Need at least 2 timesteps, got {n_t}')
    return float(cfg.total_duration_seconds) / float(n_t - 1)

def sec_to_idx(sec: float, dt: float, n_t: int) -> int:
    "Convert train seconds into bounded timestep index."
    return int(np.clip(int(round(sec / dt)), 1, n_t - 1))

def build_split(cfg: Config, train_seconds: float, n_t: int) -> Optional[SplitDef]:
    "Build (train_end,val_end,test_end); skip invalid settings safely."
    tr_raw = sec_to_idx(train_seconds, infer_dt(cfg, n_t), n_t)
    if tr_raw < max(cfg.tin + 1, 4): return None
    val_len = max(1, int(tr_raw * cfg.val_fraction_of_train))
    train_end = tr_raw - val_len; val_end = tr_raw; test_end = n_t
    if train_end - cfg.tin < 1: return None
    if cfg.predict_rest and test_end <= val_end: return None
    return SplitDef(train_end=train_end, val_end=val_end, test_end=test_end, train_seconds=float(train_seconds))

def split_hash(cfg: Config, s: SplitDef) -> str:
    "Build split hash for cache keying and stale-cache avoidance."
    p = {'train_end': s.train_end, 'val_end': s.val_end, 'test_end': s.test_end, 'train_seconds': s.train_seconds, 'tin': cfg.tin, 'tout': cfg.tout, 'predict_rest': cfg.predict_rest, 'forecast_mode': cfg.forecast_mode}
    return hashlib.sha256(json.dumps(p, sort_keys=True).encode()).hexdigest()[:16]

DT = infer_dt(cfg, N_TIMESTEPS)
split_defs = [s for s in (build_split(cfg, t, N_TIMESTEPS) for t in cfg.train_seconds_list) if s is not None]
print('dt:', DT)
print('splits:', [asdict(s) for s in split_defs])


## Section E — Normalization


In [ ]:
def fit_std_train(mm: np.memmap, train_end: int, chunk_rows: int) -> Tuple[float,float]:
    "Fit train-only mean/std by chunked streaming."
    n,_ = mm.shape; c=0; s=0.0; ss=0.0
    for i in tqdm(range(0,n,chunk_rows), desc='fit_mean_std'):
        j=min(n,i+chunk_rows); x=np.asarray(mm[i:j,:train_end], dtype=np.float64)
        c += x.size; s += x.sum(); ss += (x*x).sum()
    m = s/c; v = max(ss/c - m*m, 1e-12)
    return float(m), float(math.sqrt(v))

def fit_robust_train(mm: np.memmap, train_end: int, sample_cells: int=200_000) -> Tuple[float,float]:
    "Fit approximate train-only median/IQR using sampled cells."
    n,_ = mm.shape; idx = np.linspace(0,n-1,min(sample_cells,n),dtype=np.int64)
    vals = np.asarray(mm[idx,:train_end], dtype=np.float64).reshape(-1)
    med=float(np.median(vals)); q1,q3=np.percentile(vals,[25,75])
    return med, float(max(q3-q1,1e-6))

def normalize_memmap(mm: np.memmap, out_path: str, center: float, scale: float, chunk_rows: int, dtype: Any) -> np.memmap:
    "Normalize full timeline to memmap by chunks."
    n,t = mm.shape; out=np.memmap(out_path, mode='w+', dtype=dtype, shape=(n,t))
    for i in tqdm(range(0,n,chunk_rows), desc=f'norm->{Path(out_path).name}'):
        j=min(n,i+chunk_rows); out[i:j]=((np.asarray(mm[i:j],dtype=np.float64)-center)/scale).astype(dtype)
    out.flush(); return out

def prepare_normalized(var_name: str, mm: np.memmap, cfg: Config, split: SplitDef, cfg_h: str) -> Dict[str,Any]:
    "Compute/save train-only stats and normalized memmap for one variable."
    sk=split_hash(cfg, split); mode='robust' if cfg.use_robust_scaling else 'standard'
    key=cache_key('norm', cfg_h, sk, var_name, mode); d=Path(cfg.cache_dir)/f'norm_{key}'; d.mkdir(parents=True, exist_ok=True)
    sp=d/f'{var_name}_stats.json'; npth=d/f'{var_name}_norm.mm'
    if (not sp.exists()) or (not npth.exists()):
        if cfg.use_robust_scaling: c,s=fit_robust_train(mm, split.train_end); st='median_iqr'
        else: c,s=fit_std_train(mm, split.train_end, cfg.chunk_rows); st='mean_std'
        normalize_memmap(mm, str(npth), c, s, cfg.chunk_rows, mm_dtype(cfg))
        sp.write_text(json.dumps({'variable':var_name,'center':c,'scale':s,'stat_type':st,'cfg_hash':cfg_h,'split_hash':sk,'train_end':split.train_end,'shape':list(mm.shape)}, indent=2))
    st=json.loads(sp.read_text()); n,t=mm.shape
    nmm=np.memmap(str(npth), mode='r', dtype=mm_dtype(cfg), shape=(n,t))
    return {'stats':st, 'path':str(npth), 'mm':nmm}


## Section F — Compression module (fix prior PCA issues)


In [ ]:
def choose_modes(evr: np.ndarray, cfg: Config) -> int:
    "Choose n_modes from fixed count or explained_variance_threshold with cap."
    if cfg.explained_variance_threshold is None: return int(min(cfg.n_modes, cfg.max_modes_cap, len(evr)))
    k = int(np.searchsorted(np.cumsum(evr), cfg.explained_variance_threshold) + 1)
    return int(min(k, cfg.max_modes_cap, len(evr)))

def iter_time_chunks_from_cell_major(mm: np.memmap, t_end: int, batch_t: int):
    "Yield sample-major chunks Xb with shape (batch_t, N_cells) from mm shape (N_cells, T)."
    n_cells, _ = mm.shape
    for s in range(0, t_end, batch_t):
        e = min(t_end, s + batch_t)
        # mm[:, s:e] is (N_cells,batch_t), transpose to sample-major
        Xb = np.asarray(mm[:, s:e], dtype=np.float32).T
        assert Xb.shape[1] == n_cells
        yield s, e, Xb

def fit_pod(var_name: str, norm_mm: np.memmap, cfg: Config, split: SplitDef, cfg_h: str) -> Dict[str,Any]:
    "Fit POD in sample-major shape (time,cells) with streaming chunks and cache artifacts."
    sk=split_hash(cfg, split)
    key=cache_key('pod', cfg_h, sk, var_name, cfg.pca_mode, cfg.n_modes, cfg.explained_variance_threshold)
    d=Path(cfg.cache_dir)/f'pod_{key}'; d.mkdir(parents=True, exist_ok=True)
    mp=d/f'{var_name}_meta.json'; cp=d/f'{var_name}_coeff.mm'; comp=d/f'{var_name}_components.npy'; meanp=d/f'{var_name}_mean.npy'; evp=d/f'{var_name}_evr.npy'
    n_cells, t_total = norm_mm.shape

    if (not mp.exists()) or (not cp.exists()) or (not comp.exists()):
        if cfg.pca_mode == 'incremental':
            n_comp=min(cfg.max_modes_cap, cfg.n_modes, split.train_end, n_cells)
            pca=IncrementalPCA(n_components=n_comp)
            for _,_,Xb in tqdm(list(iter_time_chunks_from_cell_major(norm_mm, split.train_end, cfg.pca_batch_t)), desc=f'ipca_{var_name}'):
                pca.partial_fit(Xb)
        elif cfg.pca_mode == 'randomized':
            # randomized PCA needs full sample matrix; allow in debug/balanced only if feasible
            Xtr = np.concatenate([Xb for _,_,Xb in iter_time_chunks_from_cell_major(norm_mm, split.train_end, cfg.pca_batch_t)], axis=0)
            n_comp=min(cfg.max_modes_cap, cfg.n_modes, Xtr.shape[0], Xtr.shape[1])
            pca=PCA(n_components=n_comp, svd_solver='randomized', random_state=cfg.seed)
            pca.fit(Xtr)
            del Xtr
        else:
            raise ValueError(f'unknown pca_mode {cfg.pca_mode}')

        evr=pca.explained_variance_ratio_; k=choose_modes(evr,cfg)
        C=pca.components_[:k].astype(np.float32); M=pca.mean_.astype(np.float32); R=evr[:k].astype(np.float32)
        np.save(comp,C); np.save(meanp,M); np.save(evp,R)

        coeff=np.memmap(cp, mode='w+', dtype=np.float32, shape=(t_total,k))
        for s,e,Xb in tqdm(list(iter_time_chunks_from_cell_major(norm_mm, t_total, cfg.pca_batch_t)), desc=f'coeff_{var_name}'):
            coeff[s:e]=(Xb - M[None,:]) @ C.T
        coeff.flush(); del coeff
        mp.write_text(json.dumps({'coeff_shape':[t_total,k],'components_shape':list(C.shape),'mean_shape':list(M.shape),'cfg_hash':cfg_h,'split_hash':sk,'var':var_name}, indent=2))

    meta=json.loads(mp.read_text()); C=np.load(comp); M=np.load(meanp); R=np.load(evp)
    coeff=np.memmap(cp, mode='r', dtype=np.float32, shape=tuple(meta['coeff_shape']))

    rng=np.random.default_rng(cfg.seed); picks=rng.choice(meta['coeff_shape'][0], size=min(3,meta['coeff_shape'][0]), replace=False)
    errs=[]
    for ti in picks:
        ref=np.asarray(norm_mm[:,ti], dtype=np.float32)
        rec=coeff[ti] @ C + M
        errs.append(float(np.linalg.norm(rec-ref)/(np.linalg.norm(ref)+1e-12)))
    err=float(np.mean(errs)); print(f'{var_name} recon_rel_l2:', err)
    if err>cfg.recon_error_threshold:
        msg=f'Reconstruction error too high for {var_name}: {err:.4f} exceeds threshold {cfg.recon_error_threshold}'
        if cfg.profile=='debug':
            raise RuntimeError(msg)
        else:
            warnings.warn(msg)
    return {'meta':meta,'components':C,'mean':M,'explained_ratio':R,'coeff_mm':coeff,'coeff_path':str(cp)}


## Section G — Spatial graph builder


In [ ]:
def build_graph(coords: np.ndarray, cfg: Config, out_dir: Path) -> Dict[str,str]:
    "Build sparse graph (kNN/radius), optional mutual-kNN symmetrization, save edge_index/edge_weight."
    out_dir.mkdir(parents=True, exist_ok=True)
    eip=out_dir/'edge_index.npy'; ewp=out_dir/'edge_weight.npy'
    if eip.exists() and ewp.exists(): return {'edge_index':str(eip),'edge_weight':str(ewp)}
    n=coords.shape[0]; tree=cKDTree(coords)
    if cfg.graph_mode=='knn':
        d,idx=tree.query(coords, k=cfg.knn_k+1, workers=-1)
        src=np.repeat(np.arange(n,dtype=np.int64), cfg.knn_k); dst=idx[:,1:].reshape(-1).astype(np.int64); dist=d[:,1:].reshape(-1).astype(np.float32)
    elif cfg.graph_mode=='radius':
        pairs=tree.query_pairs(r=cfg.radius, output_type='ndarray')
        src=np.concatenate([pairs[:,0],pairs[:,1]]).astype(np.int64); dst=np.concatenate([pairs[:,1],pairs[:,0]]).astype(np.int64)
        dist=np.linalg.norm(coords[src]-coords[dst], axis=1).astype(np.float32)
    else: raise ValueError(f'unknown graph_mode {cfg.graph_mode}')
    if cfg.graph_mode=='knn' and cfg.mutual_knn:
        e=np.stack([src,dst],axis=1); re=np.stack([dst,src],axis=1)
        all_e=np.concatenate([e,re],axis=0); all_d=np.concatenate([dist,dist],axis=0)
        order=np.lexsort((all_e[:,1], all_e[:,0])); all_e=all_e[order]; all_d=all_d[order]
        uq,uid=np.unique(all_e, axis=0, return_index=True); src=uq[:,0]; dst=uq[:,1]; dist=all_d[uid]
    ew=1.0/(dist+1e-6)
    edge_dtype=np.int32 if n<=np.iinfo(np.int32).max else np.int64
    ei=np.stack([src.astype(edge_dtype), dst.astype(edge_dtype)], axis=0)
    np.save(eip,ei); np.save(ewp,ew.astype(np.float32))
    return {'edge_index':str(eip),'edge_weight':str(ewp)}

def build_node_features(coords: np.ndarray, p_t: Optional[np.ndarray], u_t: Optional[np.ndarray], v_t: Optional[np.ndarray], cfg: Config) -> np.ndarray:
    "Build node features from coords and optional dynamic/static placeholders."
    c=coords.astype(np.float32); c=(c-c.mean(0,keepdims=True))/(c.std(0,keepdims=True)+1e-6)
    feats=[c]
    if cfg.include_current_fields_in_nodes and p_t is not None and u_t is not None and v_t is not None:
        feats += [p_t[:,None].astype(np.float32), u_t[:,None].astype(np.float32), v_t[:,None].astype(np.float32)]
    if cfg.include_static_geometry_placeholder:
        feats.append(np.zeros((c.shape[0],1), dtype=np.float32))
    x=np.concatenate(feats, axis=1); assert x.shape[0]==coords.shape[0], f'Feature count mismatch: {x.shape[0]} vs coords {coords.shape[0]}'
    return x


## Section H — Spatial encoder (kept fixed across temporal models)


In [ ]:
class GraphConv(nn.Module):
    "Simple weighted message passing layer."
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__(); self.lin_self=nn.Linear(in_dim,out_dim); self.lin_nei=nn.Linear(in_dim,out_dim)
    def forward(self, x: torch.Tensor, ei: torch.Tensor, ew: torch.Tensor) -> torch.Tensor:
        src,dst=ei[0],ei[1]; msg=x[src]*ew.unsqueeze(-1)
        agg=torch.zeros_like(x); agg.index_add_(0,dst,msg)
        deg=torch.zeros(x.shape[0], device=x.device, dtype=x.dtype); deg.index_add_(0,dst,ew)
        agg=agg/(deg.unsqueeze(-1)+1e-6)
        return self.lin_self(x)+self.lin_nei(agg)

class SpatialEncoder(nn.Module):
    "GCN-like encoder returning pooled graph latent."
    def __init__(self, in_dim: int, hidden: int, latent: int, n_layers: int=2):
        super().__init__(); dims=[in_dim]+[hidden]*(n_layers-1)+[latent]
        self.layers=nn.ModuleList([GraphConv(dims[i],dims[i+1]) for i in range(len(dims)-1)])
    def forward(self, x: torch.Tensor, ei: torch.Tensor, ew: torch.Tensor) -> torch.Tensor:
        for i,layer in enumerate(self.layers):
            x=layer(x,ei,ew)
            if i < len(self.layers)-1: x=F.relu(x)
        return x.mean(0)

def prepare_latent_sequence(cfg: Config, split: SplitDef, coeffs: Dict[str,Dict[str,Any]], coords: np.ndarray, ei: np.ndarray, ew: np.ndarray, cfg_h: str) -> Dict[str,Any]:
    "Prepare latent sequence memmap once per split, shared by all temporal heads."
    sk=split_hash(cfg,split); mode='direct_pod' if cfg.direct_pod_input else 'spatial_projected'
    key=cache_key('latent',cfg_h,sk,mode); d=Path(cfg.cache_dir)/f'latent_{key}'; d.mkdir(parents=True,exist_ok=True)
    mp=d/'meta.json'; lp=d/'latent.mm'
    cp=np.asarray(coeffs['p']['coeff_mm'],dtype=np.float32); cu=np.asarray(coeffs['u']['coeff_mm'],dtype=np.float32); cv=np.asarray(coeffs['v']['coeff_mm'],dtype=np.float32)
    t_total=cp.shape[0]
    if (not mp.exists()) or (not lp.exists()):
        if cfg.direct_pod_input:
            lat=np.concatenate([cp,cu,cv],axis=1); D=lat.shape[1]
            mm=np.memmap(lp,mode='w+',dtype=np.float32,shape=(t_total,D)); mm[:]=lat; mm.flush(); del mm
        else:
            Dpod=cp.shape[1]+cu.shape[1]+cv.shape[1]; D=cfg.spatial_latent_dim
            mm=np.memmap(lp,mode='w+',dtype=np.float32,shape=(t_total,D))
            in_dim=2 + (3 if cfg.include_current_fields_in_nodes else 0) + (1 if cfg.include_static_geometry_placeholder else 0)
            enc=maybe_dp(SpatialEncoder(in_dim,cfg.spatial_hidden,D,cfg.spatial_layers).to(device)); enc.eval(); proj=nn.Linear(Dpod,D).to(device)
            ei_t=torch.from_numpy(ei.astype(np.int64)).to(device); ew_t=torch.from_numpy(ew.astype(np.float32)).to(device)
            with torch.no_grad():
                for t in tqdm(range(t_total), desc='latent_build'):
                    x=torch.from_numpy(build_node_features(coords,None,None,None,cfg)).to(device)
                    g=enc(x,ei_t,ew_t)
                    pod_t=torch.from_numpy(np.concatenate([cp[t],cu[t],cv[t]]).astype(np.float32)).to(device)
                    mm[t]=(g + proj(pod_t)).detach().cpu().numpy().astype(np.float32)
            mm.flush(); del mm,enc,proj
            gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        mp.write_text(json.dumps({'shape':[t_total,D],'mode':mode,'cfg_hash':cfg_h,'split_hash':sk}, indent=2))
    meta=json.loads(mp.read_text()); lat=np.memmap(lp,mode='r',dtype=np.float32,shape=tuple(meta['shape']))
    return {'latent_mm':lat,'latent_dim':int(meta['shape'][1]),'path':str(lp),'meta':meta}


## Section I — Temporal datasets


In [ ]:
class SlidingWindowDataset(Dataset):
    "Sliding windows over latent sequence with input (Tin,D) and target (Tout,D)."
    def __init__(self, lat_mm: np.memmap, start: int, end: int, tin: int, tout: int, stride: int=1):
        self.lat=lat_mm; self.tin=tin; self.tout=tout; self.idx=[]
        m=end-(tin+tout)
        for s in range(start,m+1,stride): self.idx.append(s)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i: int):
        s=self.idx[i]; x=np.asarray(self.lat[s:s+self.tin],dtype=np.float32); y=np.asarray(self.lat[s+self.tin:s+self.tin+self.tout],dtype=np.float32)
        assert x.ndim==2 and y.ndim==2, f'Expected 2D tensors, got x.ndim={x.ndim}, y.ndim={y.ndim}'
        return torch.from_numpy(x), torch.from_numpy(y)

def horizon_for_phase(cfg: Config, split: SplitDef, phase: str) -> int:
    "Resolve phase horizon for direct/AR and predict_rest mode."
    if cfg.forecast_mode=='direct':
        if cfg.predict_rest and phase=='test': return max(1, split.test_end - split.val_end)
        return cfg.tout
    return 1

def make_loaders(cfg: Config, split: SplitDef, lat_mm: np.memmap):
    "Build dataloaders with pinned memory/workers/persistent workers."
    ht=horizon_for_phase(cfg,split,'train'); hv=horizon_for_phase(cfg,split,'val'); hs=horizon_for_phase(cfg,split,'test')
    ds_tr=SlidingWindowDataset(lat_mm,0,split.train_end,cfg.tin,ht,cfg.stride)
    ds_va=SlidingWindowDataset(lat_mm,split.train_end,split.val_end,cfg.tin,hv,cfg.stride)
    ds_te=SlidingWindowDataset(lat_mm,split.val_end,split.test_end,cfg.tin,hs,cfg.stride)
    if len(ds_tr)==0 or len(ds_va)==0 or len(ds_te)==0: raise ValueError(f'Empty windows: train={len(ds_tr)}, val={len(ds_va)}, test={len(ds_te)}. Split boundaries: train_end={split.train_end}, val_end={split.val_end}, test_end={split.test_end}. Adjust Tin={cfg.tin}/Tout/train_seconds.')
    bsz=max(1, cfg.batch_size_per_gpu * max(torch.cuda.device_count(),1))
    kw=dict(batch_size=bsz,num_workers=cfg.num_workers,pin_memory=cfg.pin_memory,persistent_workers=(cfg.persistent_workers and cfg.num_workers>0))
    return DataLoader(ds_tr,shuffle=True,**kw), DataLoader(ds_va,shuffle=False,**kw), DataLoader(ds_te,shuffle=False,**kw), {'tout_train':ht,'tout_val':hv,'tout_test':hs}


## Section J — Temporal heads


In [ ]:
class LSTMHead(nn.Module):
    "LSTM temporal head (B,Tin,D)->(B,H,D)."
    def __init__(self, d: int, hidden: int=256, n_layers: int=2, horizon: int=16):
        super().__init__(); self.horizon=horizon; self.rnn=nn.LSTM(d,hidden,n_layers,batch_first=True,dropout=0.1); self.out=nn.Linear(hidden,d)
    def forward(self, x: torch.Tensor, horizon: Optional[int]=None) -> torch.Tensor:
        H=horizon or self.horizon; y,_=self.rnn(x); return self.out(y[:,-1:,:].repeat(1,H,1))

class TransformerHead(nn.Module):
    "Transformer temporal head (B,Tin,D)->(B,H,D)."
    def __init__(self, d: int, nhead: int=8, nlayers: int=2, horizon: int=16):
        super().__init__(); self.horizon=horizon
        if d < nhead: warnings.warn(f'TransformerHead: clamping nhead from {nhead} to {d} (d_model limitation)')
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=max(1,min(nhead,d)),dim_feedforward=4*d,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,num_layers=nlayers); self.out=nn.Linear(d,d)
    def forward(self, x: torch.Tensor, horizon: Optional[int]=None) -> torch.Tensor:
        H=horizon or self.horizon; z=self.enc(x); return self.out(z[:,-1:,:].repeat(1,H,1))

try:
    from mamba_ssm import Mamba
    HAS_MAMBA=True
except Exception:
    HAS_MAMBA=False

MAMBA_FALLBACK_WARNED=False

class BiSTMambaHead(nn.Module):
    "Bi-ST-Mamba with fallback to GRU when mamba_ssm unavailable."
    def __init__(self, d: int, horizon: int=16):
        super().__init__(); self.horizon=horizon; self.has_mamba=HAS_MAMBA
        if self.has_mamba:
            self.fwd=Mamba(d_model=d,d_state=16,d_conv=4,expand=2); self.bwd=Mamba(d_model=d,d_state=16,d_conv=4,expand=2)
        else:
            global MAMBA_FALLBACK_WARNED
            if not MAMBA_FALLBACK_WARNED:
                warnings.warn('mamba_ssm unavailable; fallback to GRU')
                MAMBA_FALLBACK_WARNED=True
            self.gru=nn.GRU(d,d,batch_first=True,bidirectional=True); self.proj=nn.Linear(2*d,d)
    def forward(self, x: torch.Tensor, horizon: Optional[int]=None) -> torch.Tensor:
        H=horizon or self.horizon
        if self.has_mamba:
            zf=self.fwd(x); zb=torch.flip(self.bwd(torch.flip(x,dims=[1])),dims=[1]); z=0.5*(zf+zb)
        else:
            z,_=self.gru(x); z=self.proj(z)
        return z[:,-1:,:].repeat(1,H,1)

def make_head(name: str, d: int, horizon: int) -> nn.Module:
    "Factory for unified temporal heads outputting (B,H,D)."
    if name=='lstm': return LSTMHead(d=d,horizon=horizon)
    if name=='transformer': return TransformerHead(d=d,horizon=horizon)
    if name=='bi_st_mamba': return BiSTMambaHead(d=d,horizon=horizon)
    raise ValueError(f'unknown model {name}')


## Section K — Training loop


In [ ]:
def gpu_mem_gb() -> float:
    "Return allocated GPU memory in GB on device 0."
    if not torch.cuda.is_available(): return 0.0
    return float(torch.cuda.memory_allocated(0)/1024**3)

def mse(pred: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
    "MSE with shape assertion."
    return F.mse_loss(pred,tgt)

def ar_rollout(model: nn.Module, x: torch.Tensor, horizon: int, teacher_forcing: float=0.0, y_true: Optional[torch.Tensor]=None) -> torch.Tensor:
    "Autoregressive rollout (B,horizon,D)."
    cur=x; outs=[]
    for t in range(horizon):
        p=model(cur,horizon=1); outs.append(p); nxt=p
        if y_true is not None and teacher_forcing>0 and torch.rand(1,device=cur.device).item()<teacher_forcing: nxt=y_true[:,t:t+1]
        cur=torch.cat([cur[:,1:],nxt],dim=1)
    return torch.cat(outs,dim=1)

def eval_loss(model: nn.Module, loader: DataLoader, cfg: Config, horizon: int, ar_mode: bool) -> float:
    "Compute mean loss on loader."
    model.eval(); ls=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            with autocast(enabled=(cfg.amp and device.type=='cuda')):
                p=ar_rollout(model,x,horizon,0.0,None) if ar_mode else model(x,horizon=horizon)
                ls.append(mse(p,y).item())
    return float(np.mean(ls)) if ls else float('inf')

def train_one_model(model_name: str, latent_dim: int, loaders: Tuple[DataLoader,DataLoader,DataLoader], cfg: Config, run_dir: Path, horizon_train: int, horizon_val: int) -> Dict[str,Any]:
    "Generic trainer with AdamW, AMP, grad accumulation, clip, scheduler, early stop."
    tr,va,_=loaders; model=maybe_dp(make_head(model_name,latent_dim,horizon_train).to(device))
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,cfg.epochs)) if cfg.scheduler=='cosine' else torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=2)
    ckpt=run_dir/f'best_{model_name}.pt'; best=float('inf'); bad=0; hist=[]; ar_mode=(cfg.forecast_mode=='autoregressive'); tf=cfg.teacher_forcing
    for ep in range(1,cfg.epochs+1):
        t0=time.time(); model.train(); opt.zero_grad(set_to_none=True); trls=[]
        for it,(x,y) in enumerate(tr,1):
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            with autocast(enabled=(cfg.amp and device.type=='cuda')):
                p=ar_rollout(model,x,horizon_train,tf,y) if ar_mode else model(x,horizon=horizon_train)
                loss=mse(p,y)/cfg.grad_accum
            scaler.scale(loss).backward()
            if it%cfg.grad_accum==0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            trls.append(loss.item()*cfg.grad_accum)
        tr_loss=float(np.mean(trls)) if trls else float('inf'); va_loss=eval_loss(model,va,cfg,horizon_val,ar_mode)
        if cfg.scheduler=='cosine': sch.step()
        else: sch.step(va_loss)
        if ar_mode: tf*=cfg.scheduled_sampling_decay
        e_t=time.time()-t0; rec={'epoch':ep,'train_loss':tr_loss,'val_loss':va_loss,'epoch_time_s':e_t,'gpu_mem_gb':gpu_mem_gb()}; hist.append(rec)
        print(f'[{model_name}] ep={ep:03d} train={tr_loss:.6f} val={va_loss:.6f} time={e_t:.1f}s gpu={rec["gpu_mem_gb"]:.2f}GB')
        if va_loss<best:
            best=va_loss; bad=0; torch.save({'model':model.state_dict(),'val_loss':best,'history':hist},ckpt)
        else:
            bad+=1
            if bad>=cfg.patience: print('early stop',model_name); break
    ck=torch.load(ckpt,map_location=device); model.load_state_dict(ck['model'])
    return {'model':model,'best_val_loss':float(ck['val_loss']),'history':hist,'ckpt_path':str(ckpt)}


## Section L — Evaluation & reconstruction


In [ ]:
def rmse(a: np.ndarray,b: np.ndarray)->float: "RMSE."; return float(np.sqrt(np.mean((a-b)**2)))
def mae(a: np.ndarray,b: np.ndarray)->float: "MAE."; return float(np.mean(np.abs(a-b)))
def relrmse(a: np.ndarray,b: np.ndarray)->float: "Relative RMSE."; return float(np.sqrt(np.mean((a-b)**2))/(np.sqrt(np.mean(b**2))+1e-12))

def reconstruct(coeff_pred: np.ndarray, coeff_true: np.ndarray, pod_meta: Dict[str,Any]) -> Tuple[np.ndarray,np.ndarray]:
    "Reconstruct physical fields from POD coeffs."
    C=pod_meta['components']; M=pod_meta['mean']
    return coeff_pred@C + M[None,:], coeff_true@C + M[None,:]

def region_masks(coords: np.ndarray) -> Dict[str,np.ndarray]:
    "Placeholder region masks near-airfoil and wake."
    n=coords.shape[0]; return {'near_airfoil':np.zeros(n,dtype=bool), 'wake':np.zeros(n,dtype=bool)}

def eval_and_reconstruct(model: nn.Module, test_loader: DataLoader, cfg: Config, horizon_test: int, coeffs: Dict[str,Dict[str,Any]], run_dir: Path) -> Dict[str,float]:
    "Evaluate latent predictions and reconstructed physical metrics; save summary memmaps."
    model.eval(); ar_mode=(cfg.forecast_mode=='autoregressive'); pa=[]; ta=[]
    t0=time.time()
    with torch.no_grad():
        for x,y in tqdm(test_loader, desc='test_eval'):
            x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            with autocast(enabled=(cfg.amp and device.type=='cuda')):
                p=ar_rollout(model,x,horizon_test,0.0,None) if ar_mode else model(x,horizon=horizon_test)
            pa.append(p.detach().cpu().numpy()); ta.append(y.detach().cpu().numpy())
    infer_t=time.time()-t0
    pred=np.concatenate(pa,axis=0); tgt=np.concatenate(ta,axis=0)
    kp=coeffs['p']['coeff_mm'].shape[1]; ku=coeffs['u']['coeff_mm'].shape[1]; kv=coeffs['v']['coeff_mm'].shape[1]
    assert pred.shape[-1] >= (kp+ku+kv), f'Latent dimension incompatible for model {model.__class__.__name__}: got {pred.shape[-1]}, need at least {kp+ku+kv} (p={kp}, u={ku}, v={kv})'
    pp,tp=pred[...,:kp],tgt[...,:kp]; pu,tu=pred[...,kp:kp+ku],tgt[...,kp:kp+ku]; pv,tv=pred[...,kp+ku:kp+ku+kv],tgt[...,kp+ku:kp+ku+kv]
    i=0
    p_pred,p_true=reconstruct(pp[i],tp[i],coeffs['p']); u_pred,u_true=reconstruct(pu[i],tu[i],coeffs['u']); v_pred,v_true=reconstruct(pv[i],tv[i],coeffs['v'])
    m={'latent_rmse':rmse(pred,tgt),'latent_mae':mae(pred,tgt),
       'p_rmse':rmse(p_pred,p_true),'p_mae':mae(p_pred,p_true),'p_relrmse':relrmse(p_pred,p_true),
       'u_rmse':rmse(u_pred,u_true),'u_mae':mae(u_pred,u_true),'u_relrmse':relrmse(u_pred,u_true),
       'v_rmse':rmse(v_pred,v_true),'v_mae':mae(v_pred,v_true),'v_relrmse':relrmse(v_pred,v_true),
       'global_rmse':float(np.mean([rmse(p_pred,p_true),rmse(u_pred,u_true),rmse(v_pred,v_true)])),
       'global_mae':float(np.mean([mae(p_pred,p_true),mae(u_pred,u_true),mae(v_pred,v_true)])),
       'global_relrmse':float(np.mean([relrmse(p_pred,p_true),relrmse(u_pred,u_true),relrmse(v_pred,v_true)])),
       'inference_time_s':infer_t}
    pmm=np.memmap(run_dir/'pred_summary.mm',mode='w+',dtype=np.float32,shape=pred.shape); tmm=np.memmap(run_dir/'target_summary.mm',mode='w+',dtype=np.float32,shape=tgt.shape)
    pmm[:]=pred; tmm[:]=tgt; pmm.flush(); tmm.flush(); del pmm,tmm
    return m


## Section M — Multi-duration experiment orchestrator


In [ ]:
def run_split(cfg: Config, split: SplitDef, cfg_h: str) -> List[Dict[str,Any]]:
    "Run one split: normalization, POD, graph, latent cache, train/eval all heads."
    sk=split_hash(cfg,split); split_dir=Path(cfg.output_dir)/f'split_{split.train_seconds:.2f}s_{sk}'; split_dir.mkdir(parents=True,exist_ok=True)
    np_=prepare_normalized('p',p_mm,cfg,split,cfg_h); nu_=prepare_normalized('u',u_mm,cfg,split,cfg_h); nv_=prepare_normalized('v',v_mm,cfg,split,cfg_h)
    pod_p=fit_pod('p',np_['mm'],cfg,split,cfg_h); pod_u=fit_pod('u',nu_['mm'],cfg,split,cfg_h); pod_v=fit_pod('v',nv_['mm'],cfg,split,cfg_h)
    coeffs={'p':pod_p,'u':pod_u,'v':pod_v}
    gk=cache_key('graph',cfg_h,sk,cfg.graph_mode,cfg.knn_k,cfg.radius,cfg.mutual_knn); gdir=Path(cfg.cache_dir)/f'graph_{gk}'
    g=build_graph(np.asarray(coords_mm),cfg,gdir); ei=np.load(g['edge_index']); ew=np.load(g['edge_weight'])

    rows=[]
    def run_variant(label: str, local_coeffs: Dict[str,Dict[str,Any]]) -> List[Dict[str,Any]]:
        "Train/eval all temporal heads for one variable scope."
        lat=prepare_latent_sequence(cfg,split,local_coeffs,np.asarray(coords_mm),ei,ew,cfg_h)
        tr,va,te,hm=make_loaders(cfg,split,lat['latent_mm'])
        out=[]
        for model_name in cfg.model_list:
            rk=cache_key('run',cfg_h,sk,label,model_name,cfg.forecast_mode)
            run_dir=split_dir/f'{label}_{model_name}_{rk}'; run_dir.mkdir(parents=True,exist_ok=True)
            t0=time.time(); trained=train_one_model(model_name,lat['latent_dim'],(tr,va,te),cfg,run_dir,hm['tout_train'],hm['tout_val']); train_t=time.time()-t0
            met=eval_and_reconstruct(trained['model'],te,cfg,hm['tout_test'],local_coeffs,run_dir)
            pd.DataFrame(trained['history']).to_csv(run_dir/'history.csv',index=False)
            out.append({'split_train_seconds':split.train_seconds,'split_train_end':split.train_end,'split_val_end':split.val_end,'split_test_end':split.test_end,
                        'model':model_name,'training_mode':cfg.training_mode,'variable_scope':label,'forecast_mode':cfg.forecast_mode,'latent_dim':lat['latent_dim'],
                        'params_count':int(sum(p.numel() for p in trained['model'].parameters())),'best_val_loss':trained['best_val_loss'],'train_time_s':train_t,**met})
            del trained; gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        return out

    if cfg.training_mode=='joint':
        rows.extend(run_variant('joint', coeffs))
    elif cfg.training_mode=='per_variable':
        for v in ['p','u','v']:
            c={'p':coeffs[v],'u':coeffs[v],'v':coeffs[v]}
            rows.extend(run_variant(v, c))
    else: raise ValueError(f'unknown training_mode {cfg.training_mode}')
    return rows

def run_experiments(cfg: Config) -> pd.DataFrame:
    "Loop over train_seconds_list and selected models; export summary CSV files."
    cfg_h=config_hash(cfg)
    splits=[s for s in (build_split(cfg,t,N_TIMESTEPS) for t in cfg.train_seconds_list) if s is not None]
    if not splits: raise RuntimeError('no valid splits')
    rows=[]
    for s in splits:
        print(f'\n=== split train={s.train_seconds}s ===')
        rows.extend(run_split(cfg,s,cfg_h))
    df=pd.DataFrame(rows); out=Path(cfg.output_dir); out.mkdir(parents=True,exist_ok=True)
    df.to_csv(out/'summary_master.csv',index=False)
    for sec,g in df.groupby('split_train_seconds'):
        g.to_csv(out/f'metrics_{sec:.2f}s.csv',index=False)
    print('saved', out/'summary_master.csv')
    return df

# summary_df = run_experiments(cfg)


## Section N — Visualization


In [ ]:
import matplotlib.pyplot as plt

def plot_training_curves(run_dir: Path) -> None:
    "Plot train/val curves from history.csv."
    p=run_dir/'history.csv'
    if not p.exists(): return
    h=pd.read_csv(p)
    plt.figure(figsize=(6,4)); plt.plot(h['epoch'],h['train_loss'],label='train'); plt.plot(h['epoch'],h['val_loss'],label='val')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.tight_layout(); plt.savefig(run_dir/'curve_loss.png',dpi=150); plt.close()

def plot_horizon_rmse(summary_df: pd.DataFrame, out_dir: Path) -> None:
    "Plot global RMSE per model across train durations."
    if summary_df.empty: return
    plt.figure(figsize=(8,4))
    for sec,g in summary_df.groupby('split_train_seconds'):
        plt.plot(g['model'], g['global_rmse'], marker='o', label=f'train={sec}s')
    plt.ylabel('global RMSE'); plt.xticks(rotation=20); plt.legend(); plt.tight_layout(); plt.savefig(out_dir/'horizon_rmse.png', dpi=150); plt.close()

def plot_sample_reconstructed_map(coeffs: Dict[str,Dict[str,Any]], out_dir: Path) -> None:
    "Plot sample reconstructed pressure map."
    out_dir.mkdir(parents=True,exist_ok=True)
    pmeta=coeffs.get('p');
    if pmeta is None: return
    field=np.asarray(pmeta['coeff_mm'])[0] @ pmeta['components'] + pmeta['mean']
    plt.figure(figsize=(5,4)); plt.scatter(coords_mm[:,0],coords_mm[:,1],c=field,s=1); plt.colorbar(label='pressure (norm)')
    plt.tight_layout(); plt.savefig(out_dir/'sample_spatial_map.png',dpi=150); plt.close()

def plot_probe_timeseries(out_dir: Path, probe_idx: int=0) -> None:
    "Plot p/u/v probe timeseries."
    out_dir.mkdir(parents=True,exist_ok=True)
    plt.figure(figsize=(8,4))
    for name,mm in [('p',p_mm),('u',u_mm),('v',v_mm)]:
        if probe_idx < mm.shape[0]: plt.plot(mm[probe_idx], label=name)
    plt.xlabel('timestep'); plt.ylabel('value'); plt.legend(); plt.tight_layout(); plt.savefig(out_dir/'probe_timeseries.png',dpi=150); plt.close()


## Section O — Final report cell


In [ ]:
def final_report(summary_df: pd.DataFrame) -> None:
    "Print ranked model tables per train duration and final recommendation."
    if summary_df is None or summary_df.empty:
        print('No results found. Run experiments first.'); return
    print('\n=== Ranked models per train duration (global_relrmse) ===')
    for sec,g in summary_df.groupby('split_train_seconds'):
        ranked=g.sort_values('global_relrmse',ascending=True)
        print(f'\ntrain duration: {sec}s')
        display(ranked[['model','variable_scope','global_rmse','global_mae','global_relrmse','train_time_s','inference_time_s']])
    best=summary_df.sort_values('global_relrmse',ascending=True).iloc[0]
    print('\nBest overall:'); print(best[['split_train_seconds','model','variable_scope','global_relrmse','global_rmse','train_time_s','inference_time_s']])
    long_best=summary_df.groupby('model',as_index=False).mean(numeric_only=True).sort_values('global_relrmse',ascending=True).iloc[0]
    print('\nBest by long-horizon metric proxy:'); print(long_best[['model','global_relrmse','global_rmse']])
    print('\nRecommended config for full retraining:')
    print({'profile':'full','train_seconds':float(best['split_train_seconds']),'model':str(best['model']),'training_mode':cfg.training_mode,'forecast_mode':cfg.forecast_mode,'n_modes':cfg.n_modes,'Tin':cfg.tin,'Tout':cfg.tout,'batch_size_per_gpu':cfg.batch_size_per_gpu,'epochs':cfg.epochs})

# summary_df = pd.read_csv(Path(cfg.output_dir) / 'summary_master.csv')
# final_report(summary_df)
